### TF-IDF + Ridge Baseline for Multilabel Reappraisal Classification

- SITUATION + REAPPRAISAL as text input
- annotator vote fractions as soft labels
-  equal above 0.6 as the binary label threshold for evaluation
- TF-IDF word 1-2 grams
- multi-output ridge regression
- per-label threshold tuning on the validation set

In [ ]:
 %pip install pandas numpy scipy scikit-learn iterative-stratification


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    hamming_loss,
    multilabel_confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline




In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = next(
    candidate
    for candidate in [CURRENT_DIR, *CURRENT_DIR.parents, Path("/")]
    if (candidate / "data" / "annotate-lime.csv").exists()
)

DATA_DIR = PROJECT_DIR / 'data'

OUTPUT_DIR = PROJECT_DIR / 'Code' / 'TF_IDF' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REAPPRAISAL_TECHNIQUES = ["[GM]", "[PR]", "[A]", "[EJ]", "[N]", "[UC]", "[V]", "[H]"]
LABEL_COLS = [tech.strip("[]") for tech in REAPPRAISAL_TECHNIQUES]

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

LABEL_COLS



### Load Texts and Build Soft Labels

Each annotation file has annotators as rows and Q1-Q606 as columns. We average annotator votes per item, so each label is a value between 0 and 1.

In [ ]:
texts = pd.read_csv(DATA_DIR / "annotate-lime.csv")

target = pd.DataFrame()
vote_counts = pd.DataFrame()

for tech in REAPPRAISAL_TECHNIQUES:
    label = tech.strip("[]")
    ann = pd.read_csv(DATA_DIR / f"annotations-only-{tech}.csv", sep=";")
    ann = ann.apply(pd.to_numeric, errors="coerce")
    
    target[label] = ann.mean(axis=0).values
    vote_counts[f"{label}_n_votes"] = ann.count(axis=0).values

dataset = pd.concat(
    [
        texts[["key", "SITUATION", "REAPPRAISAL"]].reset_index(drop=True),
        target.reset_index(drop=True),
        vote_counts.reset_index(drop=True),
    ],
    axis=1,
)

dataset.head()


### Label Balance

`0.6` weak annotator majority: usually 3/5 or 4/6 votes.

In [ ]:
LABEL_THRESHOLD = 0.6

y_soft = dataset[LABEL_COLS].astype("float32")
y_binary = (y_soft >= LABEL_THRESHOLD).astype(int)

print("Dataset shape:", dataset.shape)
print("\nPositive counts at threshold >= 0.6:")
display(y_binary.sum().sort_values(ascending=False).to_frame("n_positive"))

print("\nMean soft label prevalence:")
display(y_soft.mean().sort_values(ascending=False).to_frame("mean_vote_fraction"))

print("\nNumber of labels per item:")
display(y_binary.sum(axis=1).value_counts().sort_index().to_frame("n_items"))


### Create Train / Validation / Test Splits

iterative multilabel stratification to preserve the distribution of strategies.

Split sizes:

- 70% train
- 15% validation
- 15% test

In [ ]:
RANDOM_STATE = 42

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=RANDOM_STATE,
    )
    train_idx, temp_idx = next(splitter.split(dataset, y_binary))
    
    train_df = dataset.iloc[train_idx].reset_index(drop=True)
    temp_df = dataset.iloc[temp_idx].reset_index(drop=True)
    
    splitter2 = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=RANDOM_STATE,
    )
    val_idx, test_idx = next(
        splitter2.split(temp_df, (temp_df[LABEL_COLS] >= LABEL_THRESHOLD).astype(int))
    )
    
    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)
    print("Used iterative multilabel stratification.")

except ImportError:
    print("iterative-stratification is not installed. Falling back to random split.")
    train_df, temp_df = train_test_split(
        dataset,
        test_size=0.30,
        random_state=RANDOM_STATE,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_STATE,
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

for split_name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"\n{split_name}: {len(split_df)} rows")
    print((split_df[LABEL_COLS] >= LABEL_THRESHOLD).sum())


### text input

Baseline input -> situation + reappraisal. The situation gives context; the reappraisal contains most of the strategy signal.

In [ ]:
def build_text(df, text_mode="pair"):
    situation = df["SITUATION"].fillna("").astype(str)
    reappraisal = df["REAPPRAISAL"].fillna("").astype(str)
    
    if text_mode == "reappraisal":
        return reappraisal
    if text_mode == "pair":
        return "situation: " + situation + " reappraisal: " + reappraisal
    raise ValueError("text_mode must be 'pair' or 'reappraisal'")

TEXT_MODE = "pair"

X_train = build_text(train_df, TEXT_MODE)
X_val = build_text(val_df, TEXT_MODE)
X_test = build_text(test_df, TEXT_MODE)

y_train_soft = train_df[LABEL_COLS].astype("float32")
y_val_binary = (val_df[LABEL_COLS] >= LABEL_THRESHOLD).astype(int)
y_test_binary = (test_df[LABEL_COLS] >= LABEL_THRESHOLD).astype(int)

X_train.iloc[0]


## Step 8: Train TF-IDF + Ridge Regression

Ridge predicts one continuous score per label. Since the training target is the annotator vote fraction, this is a natural fit.

In [ ]:
def make_tfidf_ridge_model(alpha=10.0, max_features=20000, ngram_range=(1, 2), min_df=2):
    return make_pipeline(
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=0.95,
            max_features=max_features,
            sublinear_tf=True,
            norm="l2",
        ),
        Ridge(alpha=alpha, random_state=RANDOM_STATE),
    )


BASELINE_RIDGE_CONFIG = {
    "alpha": 10.0,
    "max_features": 20000,
    "ngram_range": (1, 2),
    "min_df": 2,
}

model = make_tfidf_ridge_model(**BASELINE_RIDGE_CONFIG)
model.fit(X_train, y_train_soft)

val_scores = np.clip(model.predict(X_val), 0, 1)
test_scores = np.clip(model.predict(X_test), 0, 1)

val_scores[:3]



### Helpers

These helpers are defined before model tuning because the tuning loop evaluates each candidate with validation-set threshold optimization.


In [ ]:
def tune_thresholds(y_true, y_score, label_cols, grid=np.arange(0.05, 0.96, 0.05)):
    thresholds = {}
    
    for i, label in enumerate(label_cols):
        best_threshold = 0.5
        best_f1 = -1
        
        for threshold in grid:
            y_pred_label = (y_score[:, i] >= threshold).astype(int)
            score = f1_score(y_true.iloc[:, i], y_pred_label, zero_division=0)
            
            if score > best_f1:
                best_f1 = score
                best_threshold = threshold
        
        thresholds[label] = float(best_threshold)
    
    return thresholds


def apply_thresholds(scores, thresholds, label_cols):
    threshold_array = np.array([thresholds[label] for label in label_cols])
    return (scores >= threshold_array).astype(int)



### Tune TF-IDF + Ridge Hyperparameters

Tuning representation and regularization strength using validation macro F1 after per-label threshold tuning.


In [ ]:
def score_ridge_config(config):
    candidate = make_tfidf_ridge_model(**config)
    candidate.fit(X_train, y_train_soft)
    candidate_val_scores = np.clip(candidate.predict(X_val), 0, 1)
    candidate_thresholds = tune_thresholds(y_val_binary, candidate_val_scores, LABEL_COLS)
    candidate_val_pred = apply_thresholds(candidate_val_scores, candidate_thresholds, LABEL_COLS)
    return {
        **config,
        "validation_micro_f1": f1_score(y_val_binary, candidate_val_pred, average="micro", zero_division=0),
        "validation_macro_f1": f1_score(y_val_binary, candidate_val_pred, average="macro", zero_division=0),
        "thresholds": candidate_thresholds,
        "model": candidate,
    }


ridge_param_grid = [
    {"alpha": alpha, "max_features": max_features, "ngram_range": ngram_range, "min_df": min_df}
    for alpha in [0.1, 1.0, 3.0, 10.0, 30.0, 100.0]
    for max_features in [10000, 20000, 50000]
    for ngram_range in [(1, 1), (1, 2), (1, 3)]
    for min_df in [1, 2, 3]
]

ridge_tuning_results = [score_ridge_config(config) for config in ridge_param_grid]
ridge_tuning_df = pd.DataFrame([
    {k: v for k, v in result.items() if k not in {"thresholds", "model"}}
    for result in ridge_tuning_results
]).sort_values("validation_macro_f1", ascending=False)

ridge_tuning_df.head(10)



### Refit best tuned ridge model

The best configuration is selected only on validation performance. The test set remains untouched until final evaluation.


In [ ]:
best_ridge_result = max(ridge_tuning_results, key=lambda result: result["validation_macro_f1"])
model = best_ridge_result["model"]
thresholds = best_ridge_result["thresholds"]

best_ridge_config = {
    key: best_ridge_result[key]
    for key in ["alpha", "max_features", "ngram_range", "min_df"]
}

val_scores = np.clip(model.predict(X_val), 0, 1)
test_scores = np.clip(model.predict(X_test), 0, 1)

print("Best Ridge configuration:")
print(json.dumps({**best_ridge_config, "ngram_range": list(best_ridge_config["ngram_range"])}, indent=2))
print("\nValidation macro F1:", best_ridge_result["validation_macro_f1"])
print("\nTuned thresholds:")
print(json.dumps(thresholds, indent=2))



### Tune One Decision Threshold Per Label
To not force every label to use threshold 0.5. Rare labels often need lower thresholds.

In [ ]:
def tune_thresholds(y_true, y_score, label_cols, grid=np.arange(0.05, 0.96, 0.05)):
    thresholds = {}
    
    for i, label in enumerate(label_cols):
        best_threshold = 0.5
        best_f1 = -1
        
        for threshold in grid:
            y_pred_label = (y_score[:, i] >= threshold).astype(int)
            score = f1_score(y_true.iloc[:, i], y_pred_label, zero_division=0)
            
            if score > best_f1:
                best_f1 = score
                best_threshold = threshold
        
        thresholds[label] = float(best_threshold)
    
    return thresholds

thresholds = tune_thresholds(y_val_binary, val_scores, LABEL_COLS)
thresholds


Evaluate the Baseline

In [ ]:
def apply_thresholds(scores, thresholds, label_cols):
    threshold_array = np.array([thresholds[label] for label in label_cols])
    return (scores >= threshold_array).astype(int)


def evaluate_multilabel(name, y_true, scores, thresholds, label_cols):
    y_pred = apply_thresholds(scores, thresholds, label_cols)
    
    metrics = {
        "split": name,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "samples_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }
    
    print(f"\n{name.upper()} METRICS")
    print(json.dumps(metrics, indent=2))
    
    print(f"\n{name.upper()} CLASSIFICATION REPORT")
    print(classification_report(y_true, y_pred, target_names=label_cols, zero_division=0))
    
    ap = average_precision_score(y_true, scores, average=None)
    ap_df = pd.DataFrame({"label": label_cols, "average_precision": ap})
    display(ap_df.sort_values("average_precision", ascending=False))
    
    return metrics, y_pred

val_metrics, val_pred = evaluate_multilabel("validation", y_val_binary, val_scores, thresholds, LABEL_COLS)
test_metrics, test_pred = evaluate_multilabel("test", y_test_binary, test_scores, thresholds, LABEL_COLS)


### Comparing Against Simple Baselines

In [ ]:
train_prevalence = (train_df[LABEL_COLS] >= LABEL_THRESHOLD).mean().values
prevalence_scores = np.tile(train_prevalence, (len(test_df), 1))

prevalence_thresholds = {label: 0.5 for label in LABEL_COLS}
prevalence_metrics, prevalence_pred = evaluate_multilabel(
    "test prevalence baseline",
    y_test_binary,
    prevalence_scores,
    prevalence_thresholds,
    LABEL_COLS,
)


In [ ]:
model_comparison_df = pd.DataFrame([
    prevalence_metrics,
    test_metrics,
]).assign(
    model=[
        "Prevalence baseline",
        "TF-IDF + tuned Ridge",
    ]
)

model_comparison_df[["model", "micro_f1", "macro_f1", "samples_f1", "hamming_loss"]]


### Visualize Models

These plots make the class imbalance, threshold behavior, per-label performance, precision-recall tradeoffs, and model comparison visible.


In [ ]:
plt.style.use("default")

label_counts = y_binary.sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(label_counts.index, label_counts.values, color="steelblue")
plt.title("Positive Label Counts at Threshold >= 0.6")
plt.xlabel("Label")
plt.ylabel("Number of Positive Examples")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()



In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_test_binary,
    test_pred,
    zero_division=0,
)

per_label_df = pd.DataFrame({
    "label": LABEL_COLS,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
}).sort_values("f1", ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(per_label_df["label"], per_label_df["f1"], color="seagreen")
plt.title("TF-IDF + Ridge Test F1 by Label")
plt.xlabel("Label")
plt.ylabel("F1 Score")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

per_label_df



In [ ]:
cms = multilabel_confusion_matrix(y_test_binary, test_pred)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.ravel()

for i, label in enumerate(LABEL_COLS):
    ax = axes[i]
    im = ax.imshow(cms[i], cmap="Blues")
    for row in range(2):
        for col in range(2):
            ax.text(col, row, str(cms[i][row, col]), ha="center", va="center", color="black")
    ax.set_title(label)
    ax.set_xticks([0, 1], labels=["Pred 0", "Pred 1"])
    ax.set_yticks([0, 1], labels=["True 0", "True 1"])

fig.suptitle("TF-IDF + Ridge Multilabel Confusion Matrices", y=1.02)
plt.tight_layout()
plt.show()



In [ ]:
comparison_plot_df = model_comparison_df.melt(
    id_vars="model",
    value_vars=["micro_f1", "macro_f1", "samples_f1"],
    var_name="metric",
    value_name="score",
)

metrics = ["micro_f1", "macro_f1", "samples_f1"]
models = model_comparison_df["model"].tolist()
x = np.arange(len(metrics))
width = 0.8 / len(models)

plt.figure(figsize=(8, 4))
for model_idx, model_name in enumerate(models):
    values = [
        model_comparison_df.loc[model_comparison_df["model"] == model_name, metric].iloc[0]
        for metric in metrics
    ]
    offset = (model_idx - (len(models) - 1) / 2) * width
    plt.bar(x + offset, values, width=width, label=model_name)

plt.title("Test Performance: Prevalence vs TF-IDF Ridge")
plt.xlabel("Metric")
plt.ylabel("Score")
plt.xticks(x, metrics)
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.25)
plt.legend(title="Model", loc="best")
plt.tight_layout()
plt.show()


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME_MAP = {
    "Prevalence baseline": "prevalence_baseline",
    "TF-IDF + tuned Ridge": "tfidf_tuned_ridge",
}

MODEL_FAMILY_MAP = {
    "Prevalence baseline": "Prevalence",
    "TF-IDF + tuned Ridge": "TF-IDF",
}

model_summary = model_comparison_df.copy()
model_summary["model_id"] = model_summary["model"].map(MODEL_NAME_MAP)
model_summary["model_name"] = model_summary["model"]
model_summary["family"] = model_summary["model"].map(MODEL_FAMILY_MAP)
model_summary["comparison_target"] = np.where(
    model_summary["model"].eq("Prevalence baseline"),
    f"vote_fraction >= {LABEL_THRESHOLD}",
    f"vote_fraction >= {LABEL_THRESHOLD} / validation-tuned",
)
model_summary["macro_mae"] = np.nan
model_summary["mean_pearson"] = np.nan
model_summary["mean_spearman"] = np.nan
model_summary = model_summary[[
    "family", "model_id", "model_name", "comparison_target", "micro_f1", "macro_f1",
    "samples_f1", "hamming_loss", "macro_mae", "mean_pearson", "mean_spearman",
]]
model_summary.to_csv(OUTPUT_DIR / "tfidf_model_summary.csv", index=False)


def make_per_label_metrics(model_name, y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        zero_division=0,
    )
    return pd.DataFrame({
        "model_id": MODEL_NAME_MAP[model_name],
        "model_name": model_name,
        "label": LABEL_COLS,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })

per_label_metrics = pd.concat([
    make_per_label_metrics("Prevalence baseline", y_test_binary, prevalence_pred),
    make_per_label_metrics("TF-IDF + tuned Ridge", y_test_binary, test_pred),
], ignore_index=True)
per_label_metrics.to_csv(OUTPUT_DIR / "tfidf_per_label_metrics.csv", index=False)


def make_average_precision(model_name, y_true, scores):
    ap = average_precision_score(y_true, scores, average=None)
    return pd.DataFrame({
        "model_id": MODEL_NAME_MAP[model_name],
        "model_name": model_name,
        "label": LABEL_COLS,
        "average_precision": ap,
    })

average_precision = pd.concat([
    make_average_precision("Prevalence baseline", y_test_binary, prevalence_scores),
    make_average_precision("TF-IDF + tuned Ridge", y_test_binary, test_scores),
], ignore_index=True)
average_precision.to_csv(OUTPUT_DIR / "tfidf_average_precision.csv", index=False)

threshold_rows = []
for model_name, model_thresholds in [
    ("Prevalence baseline", prevalence_thresholds),
    ("TF-IDF + tuned Ridge", thresholds),
]:
    for label in LABEL_COLS:
        threshold_rows.append({
            "model_id": MODEL_NAME_MAP[model_name],
            "model_name": model_name,
            "label": label,
            "threshold": model_thresholds[label],
        })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(OUTPUT_DIR / "tfidf_thresholds.csv", index=False)

ridge_tuning_df.to_csv(OUTPUT_DIR / "tfidf_ridge_tuning.csv", index=False)


def compute_soft_metrics(y_true_df, scores, label_cols):
    y_true = y_true_df[label_cols].astype(float).to_numpy()
    scores = np.asarray(scores, dtype=float)
    rows = []
    for i, label in enumerate(label_cols):
        true_label = y_true[:, i]
        pred_label = scores[:, i]
        if np.std(true_label) == 0 or np.std(pred_label) == 0:
            pearson = np.nan
            spearman = np.nan
        else:
            pearson = float(np.corrcoef(true_label, pred_label)[0, 1])
            spearman = float(pd.Series(true_label).corr(pd.Series(pred_label), method="spearman"))
        rows.append({
            "model_id": "tfidf_tuned_ridge",
            "model_name": "TF-IDF + tuned Ridge",
            "label": label,
            "mae": float(np.mean(np.abs(true_label - pred_label))),
            "rmse": float(np.sqrt(np.mean((true_label - pred_label) ** 2))),
            "pearson": pearson,
            "spearman": spearman,
            "mean_true": float(np.mean(true_label)),
            "mean_pred": float(np.mean(pred_label)),
        })
    return pd.DataFrame(rows)

ridge_soft_metrics = compute_soft_metrics(test_df, test_scores, LABEL_COLS)
ridge_soft_metrics.to_csv(OUTPUT_DIR / "tfidf_ridge_soft_metrics.csv", index=False)

ridge_soft_summary = {
    "model_id": "tfidf_tuned_ridge",
    "model_name": "TF-IDF + tuned Ridge",
    "macro_mae": float(ridge_soft_metrics["mae"].mean()),
    "macro_rmse": float(ridge_soft_metrics["rmse"].mean()),
    "mean_pearson": float(ridge_soft_metrics["pearson"].mean(skipna=True)),
    "mean_spearman": float(ridge_soft_metrics["spearman"].mean(skipna=True)),
}
(OUTPUT_DIR / "tfidf_ridge_soft_summary.json").write_text(json.dumps(ridge_soft_summary, indent=2))

model_summary.loc[model_summary["model_id"].eq("tfidf_tuned_ridge"), "macro_mae"] = ridge_soft_summary["macro_mae"]
model_summary.loc[model_summary["model_id"].eq("tfidf_tuned_ridge"), "mean_pearson"] = ridge_soft_summary["mean_pearson"]
model_summary.loc[model_summary["model_id"].eq("tfidf_tuned_ridge"), "mean_spearman"] = ridge_soft_summary["mean_spearman"]
model_summary.to_csv(OUTPUT_DIR / "tfidf_model_summary.csv", index=False)

config = {
    "random_state": RANDOM_STATE,
    "label_threshold": LABEL_THRESHOLD,
    "labels": LABEL_COLS,
    "text_input": "SITUATION + REAPPRAISAL",
    "split": "multilabel-stratified 70/15/15 when iterative-stratification is installed",
    "ridge": {
        **best_ridge_config,
        "ngram_range": list(best_ridge_config["ngram_range"]),
        "thresholds": thresholds,
        "validation_macro_f1": float(best_ridge_result["validation_macro_f1"]),
    },
}
(OUTPUT_DIR / "tfidf_config.json").write_text(json.dumps(config, indent=2))

split_counts = []
for split_name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    binary = (split_df[LABEL_COLS] >= LABEL_THRESHOLD).astype(int)
    for label in LABEL_COLS:
        split_counts.append({
            "split": split_name,
            "label": label,
            "n_rows": len(split_df),
            "positive_count": int(binary[label].sum()),
            "positive_rate": float(binary[label].mean()),
            "mean_vote_fraction": float(split_df[label].mean()),
        })
pd.DataFrame(split_counts).to_csv(OUTPUT_DIR / "tfidf_split_label_counts.csv", index=False)

print(f"Saved TF-IDF outputs to {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.glob("tfidf_*")):
    print(path.name)


1. Model: TF-IDF word unigrams/trigrams + Ridge regression.
2. Input: SITUATION + REAPPRAISAL.
3. Labels: annotator vote fractions used for training; labels binarized at `>= 0.6` for evaluation.
4. Split: multilabel-stratified 70/15/15 train/validation/test.
5. Thresholding: per-label thresholds tuned on validation F1.
6. Baselines kept: prevalence baseline and TF-IDF + tuned Ridge.
7. Metrics: micro F1, macro F1, sample F1, per-label precision/recall/F1, average precision, and Ridge soft-label metrics.
